# Entrainement des modeles CamemBERT — Guide Universitaire

Ce notebook entraine (fine-tune) les modeles CamemBERT attendus par
`app/nlp/job_analyzer.py` :

1. **`camembert-ner`** — reconnaissance d'entites nommees (NER) pour extraire
   **6 types d'information** du texte brut d'une offre d'emploi :
   `METIER` (intitule du poste), `TYPE_CONTRAT` (CDI/CDD/Stage/Alternance/
   Freelance/Temps partiel), `ENTREPRISE` (l'entreprise qui recrute),
   `LOCALISATION` (lieu du poste), `COMPETENCE` (outils, langages, soft
   skills) et `DIPLOME` (niveau d'etude ou certification requis).
2. **`camembert-classification-secteur`** — classification de sequence pour
   deviner le **secteur d'activite** de l'offre (Informatique, BTP, Finance...).
3. **`camembert-classification-categorie`** — classification de sequence
   *separee* pour deviner la **categorie d'emploi** (CDI, CDD, Stage,
   Alternance, Freelance, Temps partiel).

**Pourquoi deux modeles de classification distincts plutot qu'un seul ?**
Secteur d'activite et categorie d'emploi sont deux questions independantes
sur la meme offre ("dans quel domaine ?" vs "quel type de contrat ?") : les
melanger dans un seul jeu d'etiquettes multiplierait inutilement le nombre
de classes (10 secteurs x 6 categories = 60 combinaisons possibles) alors
que deux classifieurs simples, chacun sur sa propre question, suffisent et
restent plus faciles a entrainer avec peu de donnees.

**Avant de lancer :** dans Colab, aller dans `Execution > Modifier le type
d'execution` et choisir **GPU** (T4 suffit largement).

**Important — a lire avant la soutenance :** les jeux de donnees ci-dessous
sont volontairement petits (25 phrases pour le NER, 30 pour chaque
classification). Ils suffisent a **prouver que le pipeline d'entrainement
fonctionne de bout en bout** et a produire de vrais fichiers de modele
exploitables par `job_analyzer.py`, mais **pas** a obtenir un modele
generalisable en conditions reelles — pour ca il faut des centaines
d'exemples annotes (voir section 6).

## 0. Installation des dependances

In [ ]:
!pip install -q transformers datasets seqeval accelerate evaluate


In [ ]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device disponible : {device}")
if device == "cpu":
    print("ATTENTION : pas de GPU detecte. Executer > Modifier le type d'execution > GPU.")


## 0 bis. Corpus de donnees reelles (collecte en local, pas ici)

Les jeux de donnees des sections 1, 3 et 5 sont de petites listes ecrites a
la main : elles prouvent que le pipeline d'entrainement fonctionne, mais pas
qu'il generalise (voir section 7).

**La collecte du corpus reel ne se fait pas dans ce notebook.** Elle n'a
besoin d'aucun GPU, et elle est plus simple et plus fiable en local (VSCode) :
`app/nlp/collecte_corpus.py` combine deux sources, sans dupliquer de code —

1. les offres **deja collectees dans la base Postgres de production**
   (gratuites, aucune requete reseau) ;
2. une **collecte fraiche** via les vrais clients de scraping du projet
   (`app/scraping/lefaso_client.py`, `app/scraping/ici_pe_client.py`) pour
   depasser le volume deja en base,

dedupliquees par identifiant externe, puis exportees dans un fichier
`corpus_offres_burkina_*.csv`.

Depuis la racine du projet (necessite la base Postgres accessible, comme pour
lancer l'application) :
```
python -m app.nlp.collecte_corpus
```

Le scraping LinkedIn (Playwright) n'est volontairement pas repris pour la
collecte fraiche : les adresses IP partagees de Colab et le cout de mise en
place d'un navigateur headless ne valent pas le risque de blocage anti-robot
deja documente (section 7.2, anomalie n°3 du rapport) ; en local, ce choix ne
se pose de toute facon pas puisque la collecte ne tourne pas ici.

**Etape suivante :** annoter un echantillon de ce CSV (Label Studio ou
doccano), puis uploader le fichier annote ci-dessous.

In [ ]:
# Uploade ici le CSV annote produit en local (collecte + annotation),
# ou le CSV brut si tu veux d'abord l'inspecter avant de l'annoter.
from google.colab import files
import pandas as pd

fichiers_uploades = files.upload()
nom_fichier_corpus = next(iter(fichiers_uploades))
corpus = pd.read_csv(nom_fichier_corpus)
print(f"{len(corpus)} lignes chargees depuis {nom_fichier_corpus}")
corpus.head()

## 1. Jeu de donnees NER (demonstration, 6 types d'entites)

Schema **BIO** : chaque mot recoit une etiquette.
- `O` = mot hors entite
- `B-XXX` = premier mot d'une entite de type XXX
- `I-XXX` = mot suivant, a l'interieur de la meme entite XXX

Exemple : `"Master" -> B-DIPLOME`, `"en" -> I-DIPLOME`, `"Informatique" -> I-DIPLOME`.

In [ ]:
NER_LABELS = [
    "O",
    "B-METIER", "I-METIER",
    "B-TYPE_CONTRAT", "I-TYPE_CONTRAT",
    "B-ENTREPRISE", "I-ENTREPRISE",
    "B-LOCALISATION", "I-LOCALISATION",
    "B-COMPETENCE", "I-COMPETENCE",
    "B-DIPLOME", "I-DIPLOME",
]
label2id = {label: i for i, label in enumerate(NER_LABELS)}
id2label = {i: label for i, label in enumerate(NER_LABELS)}


In [ ]:
NER_DATA = [
    (["Nous", "recherchons", "un", "D\u00e9veloppeur", "Python", "en", "CDI", "pour", "STORM", "GROUP", "\u00e0", "Ouagadougou", "."],
     ["O", "O", "O", "B-METIER", "I-METIER", "O", "B-TYPE_CONTRAT", "O", "B-ENTREPRISE", "I-ENTREPRISE", "O", "B-LOCALISATION", "O"]),
    (["Le", "poste", "de", "Comptable", "est", "bas\u00e9", "\u00e0", "Bobo-Dioulasso", ",", "en", "CDD", "de", "6", "mois", "."],
     ["O", "O", "O", "B-METIER", "O", "O", "O", "B-LOCALISATION", "O", "O", "B-TYPE_CONTRAT", "O", "O", "O", "O"]),
    (["Vous", "ma\u00eetrisez", "Python", "et", "Django", "."],
     ["O", "O", "B-COMPETENCE", "O", "B-COMPETENCE", "O"]),
    (["Une", "bonne", "connaissance", "de", "Docker", "est", "exig\u00e9e", "."],
     ["O", "O", "O", "O", "B-COMPETENCE", "O", "O", "O"]),
    (["Un", "Master", "en", "Informatique", "est", "requis", "."],
     ["O", "B-DIPLOME", "I-DIPLOME", "I-DIPLOME", "O", "O", "O"]),
    (["Nous", "cherchons", "un", "Ing\u00e9nieur", "des", "Mines", "exp\u00e9riment\u00e9", "chez", "Total", "Energies", "."],
     ["O", "O", "O", "B-METIER", "I-METIER", "I-METIER", "O", "O", "B-ENTREPRISE", "I-ENTREPRISE", "O"]),
    (["La", "ma\u00eetrise", "de", "SQL", "et", "de", "PostgreSQL", "est", "un", "plus", "."],
     ["O", "O", "O", "B-COMPETENCE", "O", "O", "B-COMPETENCE", "O", "O", "O", "O"]),
    (["Dipl\u00f4me", "de", "Licence", "en", "Gestion", "exig\u00e9", "."],
     ["O", "O", "B-DIPLOME", "I-DIPLOME", "I-DIPLOME", "O", "O"]),
    (["Le", "Charg\u00e9", "de", "projet", "ONG", "travaillera", "\u00e0", "Ouagadougou", "en", "contrat", "Freelance", "."],
     ["O", "B-METIER", "I-METIER", "I-METIER", "I-METIER", "O", "O", "B-LOCALISATION", "O", "O", "B-TYPE_CONTRAT", "O"]),
    (["Anglais", "courant", "et", "bonne", "communication", "requis", "."],
     ["B-COMPETENCE", "O", "O", "O", "B-COMPETENCE", "O", "O"]),
    (["Odixcity", "Consulting", "recrute", "un", "Responsable", "Microfinance", "en", "Alternance", "pour", "Ouagadougou", "."],
     ["B-ENTREPRISE", "I-ENTREPRISE", "O", "O", "B-METIER", "I-METIER", "O", "B-TYPE_CONTRAT", "O", "B-LOCALISATION", "O"]),
    (["Bac+5", "en", "Ing\u00e9nierie", "Informatique", "demand\u00e9", "."],
     ["B-DIPLOME", "I-DIPLOME", "I-DIPLOME", "I-DIPLOME", "O", "O"]),
    (["Comp\u00e9tences", "requises", ":", "Git", ",", "Docker", "et", "Kubernetes", "."],
     ["O", "O", "O", "B-COMPETENCE", "O", "B-COMPETENCE", "O", "B-COMPETENCE", "O"]),
    (["Poste", "d'", "Agronome", "en", "Stage", ",", "bas\u00e9", "\u00e0", "Bobo-Dioulasso", "."],
     ["O", "O", "B-METIER", "O", "B-TYPE_CONTRAT", "O", "O", "O", "B-LOCALISATION", "O"]),
    (["Un", "dipl\u00f4me", "d'", "Ing\u00e9nieur", "Civil", "est", "n\u00e9cessaire", "."],
     ["O", "O", "O", "B-DIPLOME", "I-DIPLOME", "O", "O", "O"]),
    (["Le", "Data", "Analyst", "devra", "ma\u00eetriser", "Python", "et", "le", "Machine", "Learning", "chez", "TalentPop", "."],
     ["O", "B-METIER", "I-METIER", "O", "O", "B-COMPETENCE", "O", "O", "B-COMPETENCE", "I-COMPETENCE", "O", "B-ENTREPRISE", "O"]),
    (["Master", "2", "en", "Data", "Science", "exig\u00e9", "pour", "ce", "poste", "."],
     ["B-DIPLOME", "I-DIPLOME", "I-DIPLOME", "I-DIPLOME", "I-DIPLOME", "O", "O", "O", "O", "O"]),
    (["Nous", "recherchons", "un", "Chef", "de", "projet", "Marketing", "en", "CDI", ",", "bas\u00e9", "\u00e0", "Dakar", "."],
     ["O", "O", "O", "B-METIER", "I-METIER", "I-METIER", "I-METIER", "O", "B-TYPE_CONTRAT", "O", "O", "O", "B-LOCALISATION", "O"]),
    (["Notions", "de", "HTML", ",", "CSS", "et", "Bootstrap", "souhait\u00e9es", "."],
     ["O", "O", "B-COMPETENCE", "O", "B-COMPETENCE", "O", "B-COMPETENCE", "O", "O"]),
    (["BTS", "en", "Comptabilit\u00e9", "et", "Gestion", "appr\u00e9ci\u00e9", "."],
     ["B-DIPLOME", "I-DIPLOME", "I-DIPLOME", "I-DIPLOME", "I-DIPLOME", "O", "O"]),
    (["NTU", "International", "recrute", "\u00e0", "Paris", "en", "Temps", "partiel", "."],
     ["B-ENTREPRISE", "I-ENTREPRISE", "O", "O", "B-LOCALISATION", "O", "B-TYPE_CONTRAT", "I-TYPE_CONTRAT", "O"]),
    (["Orange", "Burkina", "propose", "un", "poste", "en", "CDI", "bas\u00e9", "\u00e0", "Ouagadougou", "."],
     ["B-ENTREPRISE", "I-ENTREPRISE", "O", "O", "O", "O", "B-TYPE_CONTRAT", "O", "O", "B-LOCALISATION", "O"]),
    (["Ing\u00e9nieur", "G\u00e9nie", "Civil", "recherch\u00e9", "pour", "un", "chantier", "\u00e0", "Abidjan", "."],
     ["B-METIER", "I-METIER", "I-METIER", "O", "O", "O", "O", "O", "B-LOCALISATION", "O"]),
    (["Ce", "poste", "est", "propos\u00e9", "en", "Freelance", "chez", "STORM", "GROUP", "."],
     ["O", "O", "O", "O", "O", "B-TYPE_CONTRAT", "O", "B-ENTREPRISE", "I-ENTREPRISE", "O"]),
    (["Stage", "de", "fin", "d'", "\u00e9tudes", "bas\u00e9", "\u00e0", "Dakar", ",", "au", "sein", "de", "TalentPop", "."],
     ["B-TYPE_CONTRAT", "O", "O", "O", "O", "O", "O", "B-LOCALISATION", "O", "O", "O", "O", "B-ENTREPRISE", "O"]),
]

print(f"{len(NER_DATA)} phrases annotees chargees pour le NER ({len(NER_LABELS)} labels).")


### Tokenisation et alignement des labels

Le tokenizer CamemBERT decoupe parfois **un mot en plusieurs sous-mots**
(ex: "Kubernetes" -> "Ku", "bernetes"). Il faut donc repeter/aligner
l'etiquette du mot d'origine sur chacun de ses sous-mots, et mettre `-100`
sur les tokens speciaux (`<s>`, `</s>`) pour que la fonction de perte les
ignore pendant l'entrainement.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

NER_MODEL_BASE = "camembert-base"
ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_BASE)

def tokenize_and_align_labels(example):
    tokenized = ner_tokenizer(example["tokens"], is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    aligned_labels = []
    previous_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            aligned_labels.append(label2id[example["ner_tags"][word_id]])
        else:
            # sous-mot suivant du meme mot : on repete le label, en le
            # convertissant en I-XXX si c'etait un B-XXX (continuite de l'entite)
            tag = example["ner_tags"][word_id]
            if tag.startswith("B-"):
                tag = "I-" + tag[2:]
            aligned_labels.append(label2id[tag])
        previous_word_id = word_id
    tokenized["labels"] = aligned_labels
    return tokenized

raw_examples = [{"tokens": tokens, "ner_tags": tags} for tokens, tags in NER_DATA]
ner_dataset = Dataset.from_list(raw_examples).map(tokenize_and_align_labels)

# Petit split train/eval (les jeux de donnees demo sont trop petits pour un vrai
# split representatif, mais ca permet de voir la mecanique d'evaluation tourner)
ner_dataset = ner_dataset.train_test_split(test_size=0.2, seed=42)
ner_dataset


## 2. Entrainement du modele NER

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_MODEL_BASE, num_labels=len(NER_LABELS), id2label=id2label, label2id=label2id
)
ner_data_collator = DataCollatorForTokenClassification(tokenizer=ner_tokenizer)
seqeval = evaluate.load("seqeval")

def compute_ner_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [NER_LABELS[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [NER_LABELS[l] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
ner_training_args = TrainingArguments(
    output_dir="./resultats_entrainement/ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,          # dataset minuscule -> plus d'epoques pour converger
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
)

ner_trainer = Trainer(
    model=ner_model,
    args=ner_training_args,
    train_dataset=ner_dataset["train"],
    eval_dataset=ner_dataset["test"],
    data_collator=ner_data_collator,
    tokenizer=ner_tokenizer,
    compute_metrics=compute_ner_metrics,
)

ner_trainer.train()


In [ ]:
import os

NER_OUTPUT_DIR = "./models/camembert-ner"
os.makedirs(NER_OUTPUT_DIR, exist_ok=True)
ner_trainer.save_model(NER_OUTPUT_DIR)
ner_tokenizer.save_pretrained(NER_OUTPUT_DIR)
print(f"Modele NER sauvegarde dans {NER_OUTPUT_DIR}")


In [ ]:
# Test rapide avec une vraie phrase, pour verifier que le modele fraichement
# entraine se comporte comme attendu avant de l'exporter
from transformers import pipeline as hf_pipeline

test_ner = hf_pipeline("ner", model=NER_OUTPUT_DIR, tokenizer=NER_OUTPUT_DIR, aggregation_strategy="simple")
test_ner("Nous recherchons un Developpeur Python en CDI chez STORM GROUP a Ouagadougou, maitrisant Docker, avec un Master en Informatique.")


## 3. Jeu de donnees classification — secteur d'activite (demonstration)

Un texte d'offre entier -> un secteur. Les libelles reprennent les secteurs
deja utilises dans le reste du projet (`app/scheduler.py`, `AUTO_SEARCHES`).

In [ ]:
SECTOR_LABELS = [
    "Informatique",
    "BTP_Genie_Civil",
    "Comptabilite_Finance",
    "Mines_Industrie",
    "Agriculture",
    "ONG_Social",
    "Microfinance_Banque",
    "Marketing_Commercial",
    "Juridique",
    "Ressources_Humaines",
]
secteur_label2id = {label: i for i, label in enumerate(SECTOR_LABELS)}
secteur_id2label = {i: label for i, label in enumerate(SECTOR_LABELS)}

SECTEUR_DATA = [
    ("Nous recherchons un developpeur Python pour renforcer notre equipe technique a Ouagadougou.", "Informatique"),
    ("Poste de Data Analyst : maitrise de SQL et Python exigee, connaissance de Docker appreciee.", "Informatique"),
    ("Ingenieur logiciel experimente en developpement web, JavaScript et bases de donnees.", "Informatique"),
    ("Ingenieur Genie Civil pour la supervision de chantiers de construction a Ouagadougou.", "BTP_Genie_Civil"),
    ("Nous recrutons un architecte pour la conception de batiments publics.", "BTP_Genie_Civil"),
    ("Chef de chantier BTP experimente en travaux publics et gestion d'equipe.", "BTP_Genie_Civil"),
    ("Comptable experimente pour la gestion de la tresorerie et des declarations fiscales.", "Comptabilite_Finance"),
    ("Poste de controleur de gestion, maitrise d'Excel et des outils comptables requise.", "Comptabilite_Finance"),
    ("Responsable financier charge de l'audit interne et du reporting.", "Comptabilite_Finance"),
    ("Ingenieur des Mines pour l'exploitation d'un site minier au Burkina Faso.", "Mines_Industrie"),
    ("Geologue experimente pour des missions d'exploration miniere.", "Mines_Industrie"),
    ("Technicien en extraction miniere et securite industrielle.", "Mines_Industrie"),
    ("Agronome pour le suivi de projets agricoles et d'elevage a Bobo-Dioulasso.", "Agriculture"),
    ("Ingenieur agroalimentaire charge du controle qualite des productions.", "Agriculture"),
    ("Technicien agricole specialise en agriculture durable.", "Agriculture"),
    ("Charge de projet pour une ONG humanitaire intervenant dans le developpement communautaire.", "ONG_Social"),
    ("Coordinateur de programmes sociaux pour une association locale.", "ONG_Social"),
    ("Responsable de projets de solidarite internationale au Burkina Faso.", "ONG_Social"),
    ("Responsable microfinance charge du developpement de l'epargne et du credit.", "Microfinance_Banque"),
    ("Conseiller bancaire pour la gestion de portefeuille clients.", "Microfinance_Banque"),
    ("Analyste credit dans une institution de microfinance.", "Microfinance_Banque"),
    ("Chef de projet Marketing pour le lancement de nouvelles campagnes publicitaires.", "Marketing_Commercial"),
    ("Commercial terrain charge du developpement des ventes.", "Marketing_Commercial"),
    ("Charge de communication digitale et de relation clientele.", "Marketing_Commercial"),
    ("Juriste d'entreprise charge du contentieux et du droit des affaires.", "Juridique"),
    ("Avocat specialise en droit du travail recherche.", "Juridique"),
    ("Assistant juridique pour la redaction de contrats commerciaux.", "Juridique"),
    ("Responsable des ressources humaines charge du recrutement et de la paie.", "Ressources_Humaines"),
    ("Charge de recrutement pour accompagner la croissance de l'entreprise.", "Ressources_Humaines"),
    ("Gestionnaire RH specialise en gestion de la paie.", "Ressources_Humaines"),
]

print(f"{len(SECTEUR_DATA)} exemples charges pour la classification secteur ({len(SECTOR_LABELS)} secteurs).")


In [ ]:
SECTEUR_MODEL_BASE = "camembert-base"
secteur_tokenizer = AutoTokenizer.from_pretrained(SECTEUR_MODEL_BASE)

def tokenize_secteur(example):
    tokenized = secteur_tokenizer(example["text"], truncation=True)
    tokenized["label"] = secteur_label2id[example["label"]]
    return tokenized

secteur_examples = [{"text": text, "label": label} for text, label in SECTEUR_DATA]
secteur_dataset = Dataset.from_list(secteur_examples).map(tokenize_secteur)
secteur_dataset = secteur_dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
secteur_dataset


## 4. Entrainement du modele de classification — secteur d'activite

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

secteur_model = AutoModelForSequenceClassification.from_pretrained(
    SECTEUR_MODEL_BASE, num_labels=len(SECTOR_LABELS), id2label=secteur_id2label, label2id=secteur_label2id
)
secteur_data_collator = DataCollatorWithPadding(tokenizer=secteur_tokenizer)
accuracy_metric = evaluate.load("accuracy")

def compute_clf_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

secteur_training_args = TrainingArguments(
    output_dir="./resultats_entrainement/classification_secteur",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
)

secteur_trainer = Trainer(
    model=secteur_model,
    args=secteur_training_args,
    train_dataset=secteur_dataset["train"],
    eval_dataset=secteur_dataset["test"],
    data_collator=secteur_data_collator,
    tokenizer=secteur_tokenizer,
    compute_metrics=compute_clf_metrics,
)

secteur_trainer.train()

In [ ]:
SECTEUR_OUTPUT_DIR = "./models/camembert-classification-secteur"
os.makedirs(SECTEUR_OUTPUT_DIR, exist_ok=True)
secteur_trainer.save_model(SECTEUR_OUTPUT_DIR)
secteur_tokenizer.save_pretrained(SECTEUR_OUTPUT_DIR)
print(f"Modele de classification (secteur) sauvegarde dans {SECTEUR_OUTPUT_DIR}")


## 5. Jeu de donnees classification — categorie d'emploi (demonstration)

Deuxieme question independante posee sur la meme offre : pas "dans quel
domaine ?" mais **"quel type de contrat ?"** (CDI, CDD, Stage, Alternance,
Freelance, Temps partiel).

In [ ]:
CATEGORIE_LABELS = [
    "CDI",
    "CDD",
    "Stage",
    "Alternance",
    "Freelance",
    "Temps_Partiel",
]
categorie_label2id = {label: i for i, label in enumerate(CATEGORIE_LABELS)}
categorie_id2label = {i: label for i, label in enumerate(CATEGORIE_LABELS)}

CATEGORIE_DATA = [
    ("Nous recherchons un developpeur Python en CDI pour rejoindre notre equipe technique a Ouagadougou.", "CDI"),
    ("Poste de comptable en contrat a duree indeterminee, integration immediate.", "CDI"),
    ("Ingenieur reseaux recrute en CDI, avantages sociaux inclus.", "CDI"),
    ("Responsable marketing recherche en CDI pour un poste perenne.", "CDI"),
    ("Chef de projet recrute en CDI, statut cadre.", "CDI"),
    ("Poste de comptable en CDD de 6 mois, remplacement conge maternite.", "CDD"),
    ("Contrat a duree determinee de 3 mois pour renforcer l'equipe commerciale.", "CDD"),
    ("CDD de 12 mois pour un poste d'assistant RH.", "CDD"),
    ("Mission en CDD dans le cadre d'un surcroit d'activite saisonnier.", "CDD"),
    ("Recrutement en CDD de 4 mois pour un projet ponctuel.", "CDD"),
    ("Stage de fin d'etudes de 6 mois en marketing digital, convention obligatoire.", "Stage"),
    ("Stage conventionne de 3 mois pour un etudiant en informatique.", "Stage"),
    ("Recherche stagiaire pour un stage d'observation de 2 mois.", "Stage"),
    ("Stage pratique en comptabilite, indemnite de stage versee.", "Stage"),
    ("Offre de stage de 4 mois destinee aux etudiants en derniere annee.", "Stage"),
    ("Contrat d'alternance de 12 mois en ingenierie informatique, rythme ecole/entreprise.", "Alternance"),
    ("Poste en alternance pour preparer un BTS en gestion.", "Alternance"),
    ("Alternance de 24 mois en developpement web, contrat d'apprentissage.", "Alternance"),
    ("Recherche alternant en marketing, rythme 3 semaines entreprise et 1 semaine ecole.", "Alternance"),
    ("Contrat de professionnalisation en alternance pour un poste commercial.", "Alternance"),
    ("Mission freelance de 3 mois pour le developpement d'une application web.", "Freelance"),
    ("Recherche consultant independant pour une mission ponctuelle de 2 mois.", "Freelance"),
    ("Mission freelance en communication digitale, facturation a la prestation.", "Freelance"),
    ("Poste en freelance pour un architecte logiciel experimente.", "Freelance"),
    ("Mission de conseil freelance de courte duree pour un audit financier.", "Freelance"),
    ("Poste a temps partiel, 20 heures par semaine, pour un assistant administratif.", "Temps_Partiel"),
    ("Emploi a temps partiel de 24h par semaine pour un poste de comptable.", "Temps_Partiel"),
    ("Recherche employe a temps partiel pour un poste de secretariat.", "Temps_Partiel"),
    ("Poste a mi-temps pour un charge de communication.", "Temps_Partiel"),
    ("Contrat a temps partiel, 15 heures hebdomadaires, poste de vendeur.", "Temps_Partiel"),
]

print(f"{len(CATEGORIE_DATA)} exemples charges pour la classification categorie d'emploi ({len(CATEGORIE_LABELS)} categories).")


In [ ]:
CATEGORIE_MODEL_BASE = "camembert-base"
categorie_tokenizer = AutoTokenizer.from_pretrained(CATEGORIE_MODEL_BASE)

def tokenize_categorie(example):
    tokenized = categorie_tokenizer(example["text"], truncation=True)
    tokenized["label"] = categorie_label2id[example["label"]]
    return tokenized

categorie_examples = [{"text": text, "label": label} for text, label in CATEGORIE_DATA]
categorie_dataset = Dataset.from_list(categorie_examples).map(tokenize_categorie)
categorie_dataset = categorie_dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
categorie_dataset


## 6. Entrainement du modele de classification — categorie d'emploi

In [ ]:
categorie_model = AutoModelForSequenceClassification.from_pretrained(
    CATEGORIE_MODEL_BASE, num_labels=len(CATEGORIE_LABELS), id2label=categorie_id2label, label2id=categorie_label2id
)
categorie_data_collator = DataCollatorWithPadding(tokenizer=categorie_tokenizer)

categorie_training_args = TrainingArguments(
    output_dir="./resultats_entrainement/classification_categorie",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
)

categorie_trainer = Trainer(
    model=categorie_model,
    args=categorie_training_args,
    train_dataset=categorie_dataset["train"],
    eval_dataset=categorie_dataset["test"],
    data_collator=categorie_data_collator,
    tokenizer=categorie_tokenizer,
    compute_metrics=compute_clf_metrics,
)

categorie_trainer.train()


In [ ]:
CATEGORIE_OUTPUT_DIR = "./models/camembert-classification-categorie"
os.makedirs(CATEGORIE_OUTPUT_DIR, exist_ok=True)
categorie_trainer.save_model(CATEGORIE_OUTPUT_DIR)
categorie_tokenizer.save_pretrained(CATEGORIE_OUTPUT_DIR)
print(f"Modele de classification (categorie d'emploi) sauvegarde dans {CATEGORIE_OUTPUT_DIR}")


In [ ]:
test_secteur = hf_pipeline("text-classification", model=SECTEUR_OUTPUT_DIR, tokenizer=SECTEUR_OUTPUT_DIR)
test_categorie = hf_pipeline("text-classification", model=CATEGORIE_OUTPUT_DIR, tokenizer=CATEGORIE_OUTPUT_DIR)

texte_test = "Nous recherchons un developpeur Python en CDI maitrisant Docker et le cloud, a Ouagadougou."
print("Secteur :", test_secteur(texte_test))
print("Categorie d'emploi :", test_categorie(texte_test))


## 7. Passer a l'echelle avec de vraies donnees

Les jeux de donnees ci-dessus (25 phrases NER, 30 + 30 exemples de
classification) servent a **valider la mecanique** (tokenisation,
alignement, entrainement, sauvegarde) — pas a obtenir un modele fiable en
production. Pour un vrai fine-tuning :

1. **Collecter un corpus reel en local**, pas dans Colab : `python -m
   app.nlp.collecte_corpus` (voir section 0 bis) combine les offres deja en
   base Postgres et une collecte fraiche via les vrais clients de scraping du
   projet, puis exporte `corpus_offres_burkina_*.csv`.
2. **Annoter** ce fichier avec un outil dedie : [Label Studio](https://labelstud.io/)
   ou [doccano](https://github.com/doccano/doccano) permettent de surligner
   METIER/TYPE_CONTRAT/ENTREPRISE/LOCALISATION/COMPETENCE/DIPLOME a la
   souris et d'exporter directement au format BIO/JSON attendu ici, et de
   labelliser chaque offre par secteur et par categorie d'emploi pour les
   deux classifieurs.
3. **Uploader** le fichier annote dans la cellule de la section 0 bis.
4. Viser **quelques centaines d'exemples annotes minimum** par entite pour
   commencer a generaliser correctement (des milliers pour de bons
   resultats en production).
5. Remplacer simplement `NER_DATA` / `SECTEUR_DATA` / `CATEGORIE_DATA` par
   les vraies donnees annotees : le reste du notebook (tokenisation,
   entrainement, export) ne change pas.

Aucune etape d'annotation automatique n'est fournie ici : c'est une tache
humaine, non substituable par le scraping lui-meme.

## 8. Recuperer les modeles dans le projet local

Trois dossiers ont ete crees : `./models/camembert-ner`,
`./models/camembert-classification-secteur` et
`./models/camembert-classification-categorie`. Pour les ramener sur ta
machine, dans le meme arrangement que `app/nlp/job_analyzer.py` :

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("camembert-ner", "zip", NER_OUTPUT_DIR)
shutil.make_archive("camembert-classification-secteur", "zip", SECTEUR_OUTPUT_DIR)
shutil.make_archive("camembert-classification-categorie", "zip", CATEGORIE_OUTPUT_DIR)

files.download("camembert-ner.zip")
files.download("camembert-classification-secteur.zip")
files.download("camembert-classification-categorie.zip")

# Ensuite, en local : dezipper chaque archive dans guideUniversitaire/models/
# de sorte a obtenir guideUniversitaire/models/camembert-ner/,
# guideUniversitaire/models/camembert-classification-secteur/ et
# guideUniversitaire/models/camembert-classification-categorie/, exactement
# les chemins par defaut de JobAnalyzerPipeline dans app/nlp/job_analyzer.py.
